In [1]:
# ============================================================
# Notebook : 04_dice_intervention.ipynb
# Purpose  : Step 4 (Intervention Validity). For each modellable
#            abnormal category, train a binary classifier that predicts
#            the category occurrence from the contemporaneous sensor
#            channels, judge its reliability with the DUAL criterion
#            (AUC >= 0.70 AND precision >= 0.40 AND test positives >= 20),
#            and - for categories that pass - generate DiCE counterfactuals
#            whose editable variables are CONSTRAINED to the Step-3 causal
#            path. Single-variable and multi-variable (complex) interventions
#            are compared by standardized L2 distance; the complex form is
#            adopted only when it is materially closer. The direct emergency
#            signal (ER) is handled as a raw signal, not modelled.
# Design   : Single code cell. No absolute path is printed. A fixed random
#            seed is set for reproducibility.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, precision_score, recall_score

import dice_ml

warnings.filterwarnings("ignore")
# Silence DiCE/tqdm progress bars for clean, reproducible notebook output.
try:
    from functools import partialmethod
    import tqdm as _tqdm_mod
    import tqdm.auto as _tqdm_auto
    _tqdm_mod.tqdm.__init__ = partialmethod(_tqdm_mod.tqdm.__init__, disable=True)
    _tqdm_auto.tqdm.__init__ = partialmethod(_tqdm_auto.tqdm.__init__, disable=True)
except Exception:
    pass
SEED = 20260821
np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
paths = pd.read_pickle(DATA / "causal_paths.pkl")["causal_paths"]
FEATURES = meta["sensor_cont"]
ABN = meta["abnormal_categories"]

# ---- Reliability thresholds (paper's dual criterion) ----
AUC_MIN, PREC_MIN, POS_MIN = 0.70, 0.40, 20

# Standardization constants (for standardized intervention distance / DiCE).
mu, sd = df[FEATURES].mean(), df[FEATURES].std().replace(0, 1)


def fit_eval(cat):
    """Train RF, return clf, train_df, test_df, and reliability metrics."""
    y = df["cat_" + cat].values
    X = df[FEATURES].astype(float)
    groups = df["HH_ID"].values
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
    tr, te = next(gss.split(X, y, groups))
    clf = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=5,
                                 class_weight="balanced", random_state=SEED, n_jobs=-1)
    clf.fit(X.iloc[tr], y[tr])
    proba = clf.predict_proba(X.iloc[te])[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y[te], proba) if y[te].sum() > 0 else np.nan
    prec = precision_score(y[te], pred, zero_division=0)
    rec = recall_score(y[te], pred, zero_division=0)
    tr_df = X.iloc[tr].copy(); tr_df["target"] = y[tr]
    te_df = X.iloc[te].copy(); te_df["target"] = y[te]
    return clf, tr_df, te_df, dict(auc=auc, precision=prec, recall=rec, n_pos_test=int(y[te].sum()))


def std_dist(cf_row, base_row, vars):
    """Standardized L2 distance between a counterfactual and the query, over `vars`."""
    d = 0.0
    for v in vars:
        d += ((cf_row[v] - base_row[v]) / sd[v]) ** 2
    return float(np.sqrt(d))


rows, interv_rows = [], []
for cat in ABN:
    clf, tr_df, te_df, mtr = fit_eval(cat)
    passes = (mtr["auc"] >= AUC_MIN) and (mtr["precision"] >= PREC_MIN) and (mtr["n_pos_test"] >= POS_MIN)
    if cat == "ER":
        # Modelled for reporting only; the direct emergency signal is used
        # raw by design, so no counterfactual intervention is prepared.
        rows.append({"category": cat, **{k: round(v, 4) if isinstance(v, float) else v for k, v in mtr.items()},
                     "passes_gate": False, "handling": "raw direct signal (held)"})
        continue
    rows.append({"category": cat, **{k: round(v, 4) if isinstance(v, float) else v for k, v in mtr.items()},
                 "passes_gate": bool(passes),
                 "handling": "DiCE intervention" if passes else "held back (reliability not met)"})

    if not passes:
        continue

    # ---- Constrained DiCE on the Step-3 causal-path variables ----
    on_path = [v for v in paths.get(cat, []) if v in FEATURES]
    if not on_path:
        interv_rows.append({"category": cat, "n_path_vars": 0, "on_path": "(none)",
                            "single_dist_med": "n/a", "complex_dist_med": "n/a",
                            "adopted": "no on-path variable", "n_queries_ok": 0})
        continue
    d_int = dice_ml.Data(dataframe=tr_df, continuous_features=FEATURES, outcome_name="target")
    m_int = dice_ml.Model(model=clf, backend="sklearn")
    exp = dice_ml.Dice(d_int, m_int, method="random")

    # Query points: positive test cases.
    pos = te_df[te_df["target"] == 1]
    if len(pos) == 0:
        continue
    q = pos.drop(columns=["target"]).sample(min(30, len(pos)), random_state=SEED).reset_index(drop=True)

    single_d, complex_d, n_ok = [], [], 0
    has_complex = len(on_path) >= 2  # a complex intervention is only meaningful with >=2 on-path vars
    for i in range(len(q)):
        base = q.iloc[i]
        try:  # single-variable intervention: the strongest on-path variable only
            cf1 = exp.generate_counterfactuals(q.iloc[[i]], total_CFs=1, desired_class="opposite",
                                               features_to_vary=[on_path[0]], verbose=False)
            f1 = cf1.cf_examples_list[0].final_cfs_df
            if f1 is not None and len(f1):
                single_d.append(std_dist(f1.iloc[0], base, [on_path[0]]))
                n_ok += 1
        except Exception:
            pass
        if has_complex:
            try:  # complex intervention: all on-path variables
                cfk = exp.generate_counterfactuals(q.iloc[[i]], total_CFs=1, desired_class="opposite",
                                                   features_to_vary=on_path, verbose=False)
                fk = cfk.cf_examples_list[0].final_cfs_df
                if fk is not None and len(fk):
                    complex_d.append(std_dist(fk.iloc[0], base, on_path))
            except Exception:
                pass
    s_med = float(np.median(single_d)) if single_d else np.nan
    c_med = float(np.median(complex_d)) if complex_d else np.nan
    if not has_complex:
        adopt = "single (only one on-path variable)"
    elif c_med == c_med and s_med == s_med and c_med < 0.9 * s_med:
        adopt = "complex"
    else:
        adopt = "single"
    interv_rows.append({"category": cat, "n_path_vars": len(on_path),
                        "on_path": ", ".join(on_path),
                        "single_dist_med": round(s_med, 3) if s_med == s_med else "n/a",
                        "complex_dist_med": round(c_med, 3) if c_med == c_med else "n/a",
                        "adopted": adopt, "n_queries_ok": n_ok})

perf = pd.DataFrame(rows)
perf.to_csv(TAB / "table_model_performance.csv", index=False, encoding="utf-8-sig")
interv = pd.DataFrame(interv_rows)
interv.to_csv(TAB / "table_intervention_comparison.csv", index=False, encoding="utf-8-sig")
pd.to_pickle({"performance": perf, "pass": {r["category"]: r["passes_gate"] for r in rows}},
             DATA / "model_gate.pkl")

# ---- Console summary ----
print("=== Step 4: intervention validity ===")
print(f"Dual reliability criterion: AUC >= {AUC_MIN}, precision >= {PREC_MIN}, test positives >= {POS_MIN}")
print()
print("Per-category classifier performance and gate:")
print(perf.to_string(index=False))
print()
n_pass = int(perf["passes_gate"].sum())
print(f"Categories passing the gate (intervention strategy prepared): {n_pass} of {len(ABN)}")
print(f"  passed : {[r['category'] for r in rows if r['passes_gate']]}")
print(f"  held   : {[r['category'] for r in rows if not r['passes_gate']]}")
print()
print("Single vs complex intervention (standardized L2 distance):")
print(interv.to_string(index=False))
print()
print("Tables saved : table_model_performance, table_intervention_comparison")
print("Artifact     : model_gate.pkl")


=== Step 4: intervention validity ===
Dual reliability criterion: AUC >= 0.7, precision >= 0.4, test positives >= 20

Per-category classifier performance and gate:
category    auc  precision  recall  n_pos_test  passes_gate                        handling
      HR 0.9987     0.9630  0.9988        3314         True               DiCE intervention
      BR 0.9986     0.9209  0.9965        1997         True               DiCE intervention
      ST 0.9355     0.2881  0.5967        1448        False held back (reliability not met)
      SP 0.9997     0.7602  1.0000         168         True               DiCE intervention
      ER 0.7667     0.0000  0.0000          63        False        raw direct signal (held)

Categories passing the gate (intervention strategy prepared): 3 of 5
  passed : ['HR', 'BR', 'SP']
  held   : ['ST', 'ER']

Single vs complex intervention (standardized L2 distance):
category  n_path_vars    on_path  single_dist_med complex_dist_med                            adopte